# Checkpoint 3 — Mock Contest 3 Solutions

Reference solutions for all 7 problems. Each is a complete stdin/stdout program; the cells below are display-only mirrors of the solver files in the `assets/` folder.

## Question 1

**Notice:** `count_routes(row, col)` counts the routes from that cell to the bottom-right cell that avoid every cell already on the current route. Reaching the bottom-right cell is one complete route, which also covers the single-cell garden. Otherwise, mark the cell visited, add up the routes through each open, unvisited neighbour, and then un-mark it so that other routes may pass through it later. Without the un-marking, routes found later would be wrongly blocked. The garden has at most `25` cells, so trying every route that never repeats a cell finishes quickly.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
rows = int(tokens[0])
cols = int(tokens[1])
grid = []
visited = []
r = 0
while r < rows:
    grid.append(tokens[2 + r])
    visited_row = []
    c = 0
    while c < cols:
        visited_row.append(False)
        c = c + 1
    visited.append(visited_row)
    r = r + 1


def count_routes(row, col):
    if row == rows - 1 and col == cols - 1:
        return 1
    total = 0
    visited[row][col] = True
    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for step in steps:
        next_row = row + step[0]
        next_col = col + step[1]
        if 0 <= next_row and next_row < rows and 0 <= next_col and next_col < cols:
            if grid[next_row][next_col] == "." and not visited[next_row][next_col]:
                total = total + count_routes(next_row, next_col)
    visited[row][col] = False
    return total


print(str(count_routes(0, 0)))


## Question 2

**Notice:** push each bit with `appendleft`. For `NOT`, `popleft` one bit and push `1 - bit`. For `AND`, `OR` and `IMP`, `popleft` the RIGHT operand first and the LEFT operand second, then push the gate's result, where `left IMP right` is `0` only for `1 IMP 0`. Every token does a constant amount of stack work, so 200000 tokens take one quick pass; the one bit left on the stack is the answer.

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
tokens = data.split()
stack = deque()
i = 0
while i < len(tokens):
    token = tokens[i]
    if token == "NOT":
        value = stack.popleft()
        stack.appendleft(1 - value)
    elif token == "AND" or token == "OR" or token == "IMP":
        right = stack.popleft()
        left = stack.popleft()
        if token == "AND":
            if left == 1 and right == 1:
                value = 1
            else:
                value = 0
        elif token == "OR":
            if left == 1 or right == 1:
                value = 1
            else:
                value = 0
        else:
            if left == 1 and right == 0:
                value = 0
            else:
                value = 1
        stack.appendleft(value)
    else:
        stack.appendleft(int(token))
    i = i + 1
print(str(stack.popleft()))


## Question 3

**Notice:** repeatedly take `number % base` as the next digit (from the right) and divide with `number // base`. Prepend each digit to `forward` so the string reads in the correct order, and append it to `backward` to get the digits in reverse. Handle `0` separately as `"0"`. The representation is a palindrome exactly when `forward == backward`. Each step divides by at least 2, so even `10^18` needs only about 60 steps.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
number = int(parts[0])
base = int(parts[1])
forward = ""
backward = ""
if number == 0:
    forward = "0"
    backward = "0"
while number > 0:
    digit = str(number % base)
    forward = digit + forward
    backward = backward + digit
    number = number // base
if forward == backward:
    answer = "YES"
else:
    answer = "NO"
print(forward)
print(answer)


## Question 4

**Notice:** turn each switch into a bitmask whose bit `lamp` is set for every lamp it flips. Pressing several switches flips each lamp once per switch that touches it, and that is exactly the XOR `^` of their masks, so the order of presses does not matter. Try every choice from `0` through `(1 << S) - 1`: XOR the chosen masks, and when the result equals `(1 << L) - 1` (every lamp on), compare the number of pressed switches with the best so far. If no choice works, `best` stays `-1`. With `S` at most `16` there are only `65536` choices of at most `16` switches each, about one million steps.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
lamps = int(tokens[0])
switch_count = int(tokens[1])
switches = []
position = 2
s = 0
while s < switch_count:
    k = int(tokens[position])
    mask = 0
    j = 0
    while j < k:
        lamp = int(tokens[position + 1 + j])
        mask = mask | (1 << lamp)
        j = j + 1
    switches.append(mask)
    position = position + 1 + k
    s = s + 1

all_on = (1 << lamps) - 1
best = -1
choice = 0
while choice < (1 << switch_count):
    state = 0
    pressed = 0
    s = 0
    while s < switch_count:
        if choice & (1 << s):
            state = state ^ switches[s]
            pressed = pressed + 1
        s = s + 1
    if state == all_on:
        if best == -1 or pressed < best:
            best = pressed
    choice = choice + 1
print(str(best))


## Question 5

**Notice:** start the count at `1` for the first corner. For each leg, take `dx` and `dy` as the absolute differences of the coordinates. The integer points on the leg are evenly spaced, and there are `gcd(dx, dy)` of them after the start up to and including the end, so add that. A zero difference works, because `gcd(0, 5) = 5`, and taking absolute values handles legs that go left or down. Euclid's algorithm replaces the pair with `(b, a % b)` until the remainder is `0`, which takes only a few dozen steps even for two-billion-unit legs, so `50000` legs are fast.

In [ ]:
import sys


def gcd(a, b):
    while b != 0:
        remainder = a % b
        a = b
        b = remainder
    return a


data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
stops = 1
i = 1
while i < n:
    dx = abs(int(tokens[1 + 2 * i]) - int(tokens[1 + 2 * (i - 1)]))
    dy = abs(int(tokens[2 + 2 * i]) - int(tokens[2 + 2 * (i - 1)]))
    stops = stops + gcd(dx, dy)
    i = i + 1
print(str(stops))


## Question 6

**Notice:** run one Sieve of Eratosthenes through `N`, crossing off multiples of each prime starting at `prime * prime`. Starting at the square is safe because smaller multiples were already crossed off by smaller primes, and it must include the square itself (`169` is not prime). Then count every `low` with `low + 2 <= N` where both `low` and `low + 2` are still marked prime. The sieve does about `N log log N` work plus one linear pass, far faster than trial division on every candidate.

In [ ]:
import sys

data = sys.stdin.read()
limit = int(data.strip())
is_prime = []
i = 0
while i <= limit:
    is_prime.append(True)
    i = i + 1
is_prime[0] = False
is_prime[1] = False

prime = 2
while prime * prime <= limit:
    if is_prime[prime]:
        multiple = prime * prime
        while multiple <= limit:
            is_prime[multiple] = False
            multiple = multiple + prime
    prime = prime + 1

pairs = 0
low = 2
while low + 2 <= limit:
    if is_prime[low] and is_prime[low + 2]:
        pairs = pairs + 1
    low = low + 1
print(str(pairs))


## Question 7

**Notice:** `heaviest(node)` gives back the best root-to-leaf total starting at `node`. A leaf gives back its own value. A node with one child adds its value to that child's result, because a missing child is NOT a path worth `0`, and a path may not stop at a non-leaf. A node with two children adds its value to the larger of the two results. Each node is visited once, so the work is linear in `N`, and the height limit of 900 keeps the recursion under Python's default depth limit.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
node_count = int(parts[0])
root = int(parts[1])
values = []
left = []
right = []
i = 0
while i < node_count:
    values.append(int(parts[i * 3 + 2]))
    left.append(int(parts[i * 3 + 3]))
    right.append(int(parts[i * 3 + 4]))
    i = i + 1

def heaviest(node):
    if left[node] == -1 and right[node] == -1:
        return values[node]
    if left[node] == -1:
        return values[node] + heaviest(right[node])
    if right[node] == -1:
        return values[node] + heaviest(left[node])
    return values[node] + max(heaviest(left[node]), heaviest(right[node]))

print(str(heaviest(root)))
